## Simple tokenizer and vocabulary from *The Verdict*

Hand-built splitting, a string-to-id vocabulary, and a small encode/decode demo.

In [2]:
import urllib.request
import re
import torch

# Download the sample short story from the book repo and save it locally.
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

# Load UTF-8 text; print length and a short prefix as a sanity check.
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    print("Total characters in the dataset:", len(raw_text))
    print(raw_text[:99])

# Split on whitespace and punctuation; the capture group keeps delimiters as separate tokens.
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
# Unique token types (strings), sorted so ids are stable and comparable across runs.
all_tokens = sorted(list(set(preprocessed)))
# Document boundary token; fallback id for any string not in the vocabulary at encode time.
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
# Map each token string to a contiguous integer id for embedding tables and lookup.
vocab = {token: integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

# Inspect vocabulary size and the last few (token, id) pairs, including specials.
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

class SimpleTokenizerV2:
    """
    A simple tokenizer that encodes and decodes text using a vocabulary.
    """

    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Replace unknown strings with <|unk|> before mapping to ids.
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Strip spaces that join() inserted before punctuation.
        text = re.sub(fr'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

# Join two segments with <|endoftext|> as a document separator (multi-document style).
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1,text2))
print(text)

tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))


ModuleNotFoundError: No module named 'torch'

## GPT-2 BPE (`tiktoken`) and next-token windows

Use the real GPT-2 byte-pair tokenizer, then build one input/target pair for causal language modeling.

In [3]:
from importlib.metadata import version
import tiktoken
import torch
from torch import max_pool1d
from torch.utils.data import Dataset, DataLoader


# GPT-2 byte-pair encoding via tiktoken — same scheme as later chapters, not identical to SimpleTokenizerV2.
tokenizer = tiktoken.get_encoding("gpt2")
# Adjacent string literals concatenate; no space between "palace." and "of", so BPE splits unknown words into subwords.
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace." "of someunknownPlace"
)
# List special tokens here so encode accepts them (otherwise tiktoken may raise or omit them).
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>","<|unk|>"})
print(integers)

# Decode token ids back to a single string (spaces and subword boundaries follow BPE rules).
strings = tokenizer.decode(integers)
print(strings)

# Load the full text and encode it — preparation for sliding-window / next-token batches (not a full DataLoader yet).
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Full-book token count under the BPE tokenizer.
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

# Skip the first 50 tokens so this small demo window does not always start at the beginning of the file.
enc_sample = enc_text[50:]

context_size = 4
# x: input context; y: next-token targets (shifted by one) for causal LM — one step of a sliding window over enc_sample.
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x:  {x}")
print(f"y:      {y}")

#Since we are processing the inputs with the targets, we can create next-word prediction tasks
# for i in range (1, context_size -1):
#     context = enc_sample[:i]
#     desired = enc_sample[i]
#     print (context, "------>", desired)

# Walk prefix lengths 1..context_size over enc_sample: context = first i tokens, desired = token at index i
# (next-token target). decode() turns ids back into text for readability.
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "------>", tokenizer.decode([desired]))

# A Dataset that produces sliding-window input-target pairs for next-token prediction.
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids =[]
        self.target_ids =[]
# Encode the full text into token ids, then walk a sliding window over the list of ids to create input-target pairs.
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

ModuleNotFoundError: No module named 'torch'